# Combined Datasets — Representation 3: Native 3D Point Set (PointNet++)

Trains PyTorch **PointNet / PointNet++** on unordered 3D point matrix (5 channels: x, y, z, v, time).

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

sns.set_theme(style="whitegrid")
torch.manual_seed(42)

preproc_dir = Path("datasets/preprocessed")
if not preproc_dir.exists(): preproc_dir = Path("../datasets/preprocessed")
if not preproc_dir.exists(): preproc_dir = Path("../../datasets/preprocessed")

X_rep3 = np.load(preproc_dir / "X_rep3_pointset_combined.npy")
y_rep3 = np.load(preproc_dir / "y_rep3_pointset_combined.npy")

print(f"Loaded Representation 3 Point Set Tensor: {X_rep3.shape}")
print(f"Labels: {np.bincount(y_rep3.astype(int))} (0 = ADL, 1 = Fall)")

In [ ]:
class PointNetFallDetector(nn.Module):
    def __init__(self, in_channels=5, num_classes=2):
        super(PointNetFallDetector, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 256, 1)
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.4)
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))
        global_feat = torch.max(x, 2, keepdim=True)[0].view(x.size(0), -1)
        out = self.relu(self.fc1(self.dropout(global_feat)))
        return self.fc2(out)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PointNetFallDetector().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_rep3, y_rep3, test_size=0.2, random_state=42, stratify=y_rep3)
tr_ds = TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long))
te_ds = TensorDataset(torch.tensor(X_te, dtype=torch.float32), torch.tensor(y_te, dtype=torch.long))
tr_loader = DataLoader(tr_ds, batch_size=32, shuffle=True)
te_loader = DataLoader(te_ds, batch_size=32, shuffle=False)

epochs = 15
for ep in range(epochs):
    model.train()
    tot_loss, corr = 0, 0
    for bx, by in tr_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        tot_loss += loss.item() * bx.size(0)
        corr += (out.argmax(1) == by).sum().item()
    if (ep + 1) % 5 == 0 or ep == 0:
        print(f"Epoch {ep+1:02d}/{epochs} - Loss: {tot_loss/len(X_tr):.4f} - Acc: {corr/len(X_tr)*100:.2f}%")

In [ ]:
model.eval()
preds, probs, targets = [], [], []
with torch.no_grad():
    for bx, by in te_loader:
        bx = bx.to(device)
        out = model(bx)
        probs.extend(torch.softmax(out, dim=1)[:, 1].cpu().numpy())
        preds.extend(out.argmax(1).cpu().numpy())
        targets.extend(by.numpy())

acc = accuracy_score(targets, preds)
auc = roc_auc_score(targets, probs)
print(f"Representation 3 (PointNet) Test Accuracy: {acc*100:.2f}% | ROC-AUC: {auc:.4f}")
print("\nClassification Report:\n", classification_report(targets, preds, target_names=["ADL", "Fall"]))